In [ ]:
import os, sys
sys.path.append('../src')
from FIT_python.data_split_and_summary.datasplit_and_summary_wraper import prepare_all_splits

# Daten-Splits sind bereits vorhanden, Aufruf bei Bedarf:
#prepare_all_splits()

In [ ]:
from FIT_python.plot_style import apply_styleapply_style()import pandas as pdfrom pathlib import Pathfrom FIT_python.pipeline_individual_id.data_split import sequential_holdout_idsfrom FIT_python.pipeline_individual_id.pair_generation import build_pairwise_comparisonsfrom FIT_python.pipeline_individual_id.data_split import assign_foldsfrom FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaselinefrom FIT_python.pipeline_individual_id.supervised_umap_pipeline import run as run_umapfrom FIT_python.pipeline_individual_id.siamese_pipeline import run as run_siamesefrom FIT_python.pipeline_individual_id.evaluation import compute_confusionimport seaborn as sns, matplotlib.pyplot as pltfrom FIT_python.data_split_and_summary.data_import_utils import (    load_and_prep_df_individual,    get_feature_cols,)from FIT_python.config import RESULTS_DATA_DIR, SPLITS_DIR# %% 1) Basis-Pfade & Hyperparameter-Räume (unverändert)splits_root = Path("data/splits/eurasian_otter")models_root = (    RESULTS_DATA_DIR    / "eurasian_otter_random_search_standard_metrics"    / "best_maj_test_pct")output_root = Path("results/data/pairwise_comp_eurasian_otter")output_root.mkdir(parents=True, exist_ok=True)# 6) Loop über Speziesspecies_dir = Path(SPLITS_DIR) / "eurasian_otter"species = "eurasian_otter"print(f"\n🔄 Processing species: {species}")# 6.1) Train/Test laden & Fold entfernendf = load_and_prep_df_individual(    species, Path(SPLITS_DIR))  # falls hier bugs valid vales f und m checkenfeature_cols = [    c for c in df.columns if c.startswith(("dist", "ang", "t", "v")) and c != "trail"]ids = df["individual_id"].unique()split = sequential_holdout_ids(ids, val_sizes=[3, 5, 9], n_iter=1, random_state=0)[0]train_df = df[df["individual_id"].isin(split["train_ids"])]val_df = df[df["individual_id"].isin(split["val_ids"])]val_df["_group_id"] = val_df["individual_id"].where(val_df["individual_id"].notna() & (val_df["individual_id"] != "unknown"), val_df["trail"]).astype(str)sex_map = val_df.groupby("_group_id")["sex"].first().fillna('unknown').replace('', 'unknown').to_dict()fallback_map = val_df.groupby("_group_id")["individual_id"].first().apply(lambda x: pd.isna(x) or x=='unknown').to_dict()val_comps, _ = build_pairwise_comparisons(val_df, id_col="individual_id", sex_map=sex_map, fallback_map=fallback_map, trail_col="trail")val_comps = assign_folds(val_comps, sex_map, n_folds=3, random_state=42)# In ein DataFrame überführendf_val_comps = pd.DataFrame(val_comps)

In [ ]:
val_comps

In [ ]:
import pandas as pdimport seaborn as snsimport matplotlib.pyplot as pltfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import StratifiedKFold, cross_val_predictfrom FIT_python.pipeline_individual_id.baseline_pipeline import DistanceBaselinefrom FIT_python.pipeline_individual_id.evaluation import compute_confusionsplits = sequential_holdout_ids(ids, val_sizes=[4, 8, 12], n_iter=3, random_state=0)for split in splits:    train_df = df[df["individual_id"].isin(split["train_ids"])]    val_df   = df[df["individual_id"].isin(split["val_ids"])]    val_df['_group_id'] = val_df['individual_id'].where(val_df['individual_id'].notna() & (val_df['individual_id'] != 'unknown'), val_df['trail']).astype(str)    sex_map = val_df.groupby('_group_id')['sex'].first().fillna('unknown').replace('', 'unknown').to_dict()    fallback_map = val_df.groupby('_group_id')['individual_id'].first().apply(lambda x: pd.isna(x) or x=='unknown').to_dict()    val_comps, _ = build_pairwise_comparisons(val_df, id_col='individual_id', sex_map=sex_map, fallback_map=fallback_map, trail_col='trail')    val_comps = assign_folds(val_comps, sex_map, n_folds=2, random_state=42)    res_base = DistanceBaseline.run(train_df, val_comps, feature_cols, val_df=val_df)    Xb = res_base.filter(regex="^dist_")    yb = res_base["same_individual"].map({"True": 1, "False": 0})    cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)    lr = LogisticRegression(max_iter=200)    y_pred = cross_val_predict(lr, Xb, yb, cv=cv)    cm_base = compute_confusion(pd.DataFrame({"same_individual": yb, "pred": y_pred}))    print(f"Iteration {split['iteration']}, n_val={split['n_val']}")    display(cm_base)

In [ ]:
res_umap, emb_umap = run_umap(train_df, val_comps, feature_cols, val_df=val_df)
yu = res_umap["same_individual"].map({"True": 1, "False": 0})
pred_u = (res_umap["pred_same_proba"] > 0.5).astype(int)
res_umap["pred"] = (res_umap["pred_same_proba"] > 0.5).astype(int)
cm_umap = compute_confusion(res_umap, true_col="same_individual", pred_col="pred")

sns.heatmap(cm_umap, annot=True, fmt="d")
plt.show()

In [ ]:
# Siamese-Pipeline aktuell deaktiviert
res_siam = run_siamese(train_df, val_comps, feature_cols, val_df=val_df)
ys = pd.Series([c["same_individual"] for c in val_comps]).map({"True": 1, "False": 0})
pred_s = (pd.Series([r["pred_same"] for r in res_siam]) > 0.5).astype(int)
cm_siam = compute_confusion(pd.DataFrame({"same_individual": ys, "pred": pred_s}))
sns.heatmap(cm_siam, annot=True, fmt="d")
plt.show()